# Actividad 1. Exploración y sábana analítica

El objetivo es comprender la fuente, revisar su calidad, definir la granularidad y construir una sábana con una fila por cuenta por cobrar. La transformación principal se realiza en SQL y Python se utiliza para validar y analizar los resultados.


In [43]:
from pathlib import Path
import sqlite3
import sys

import pandas as pd

project_path = Path.cwd()
if project_path.name == "notebooks":
    project_path = project_path.parent

if str(project_path) not in sys.path:
    sys.path.insert(0, str(project_path))

from src.analytical_table_builder import AnalyticalTableBuilder
from src.database import SQLiteDatabase

database_path = project_path / "data" / "base_datos_historica.db"
database_uri = f"file:{database_path.as_posix()}?mode=ro"

if not database_path.exists():
    raise FileNotFoundError(
        f"No se encontró la base de datos: {database_path}"
    )

print("Base disponible: data/base_datos_historica.db")


Base disponible: data/base_datos_historica.db


## 1. Fuente y alcance

Se valida la integridad de la base, su estructura y los principales valores de volumen, fechas y montos.


In [44]:
with sqlite3.connect(database_uri, uri=True) as connection:
    integrity_result = connection.execute(
        "PRAGMA integrity_check;"
    ).fetchone()[0]

    database_objects = pd.read_sql_query(
        """
        SELECT
            name AS object_name,
            type AS object_type
        FROM sqlite_master
        WHERE type IN ('table', 'view')
          AND name NOT LIKE 'sqlite_%'
        ORDER BY
            type,
            name;
        """,
        connection,
    )

print(f"Integridad de la base: {integrity_result}")
database_objects

Integridad de la base: ok


,object_name,object_type
0,tabla1,table
1,v_resumen_mensual,view


In [45]:
with sqlite3.connect(database_uri, uri=True) as connection:
    table_schema = pd.read_sql_query(
        """
        PRAGMA table_info("tabla1");
        """,
        connection,
    )

table_schema

,cid,name,type,notnull,dflt_value,pk
0,0,cod_apli_prod,TEXT,0,None,0
1,1,descri_cod_apli_prod,TEXT,0,None,0
2,2,num_cta,INTEGER,0,None,0
3,3,f_creacion,INTEGER,0,None,0
4,4,f_ultimo_pago,INTEGER,0,None,0
5,5,vlr_original,REAL,0,None,0
6,6,vlr_pagado,REAL,0,None,0
7,7,vlr_pendiente_pago,REAL,0,None,0
8,8,cod_trn,INTEGER,0,None,0
9,9,descri_cod_trn,TEXT,0,None,0


In [46]:
with sqlite3.connect(database_uri, uri=True) as connection:
    source_profile = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT num_cta) AS distinct_accounts,
            COUNT(DISTINCT cod_apli_prod) AS distinct_products,
            COUNT(DISTINCT cod_trn) AS distinct_transactions,
            MIN(f_creacion) AS minimum_creation_date,
            MAX(f_creacion) AS maximum_creation_date,
            MIN(f_ultimo_pago) AS minimum_payment_date,
            MAX(f_ultimo_pago) AS maximum_payment_date,
            MIN(vlr_original) AS minimum_original_amount,
            MAX(vlr_original) AS maximum_original_amount,
            ROUND(SUM(vlr_original), 2) AS total_original_amount,
            ROUND(SUM(vlr_pagado), 2) AS total_paid_amount,
            ROUND(SUM(vlr_pendiente_pago), 2) AS total_pending_amount
        FROM tabla1;
        """,
        connection,
    )

source_profile.T

,0
total_rows,2.173900e+04
distinct_accounts,8.000000e+02
distinct_products,2.000000e+00
distinct_transactions,7.100000e+01
minimum_creation_date,2.024101e+07
maximum_creation_date,2.025081e+07
minimum_payment_date,2.024102e+07
maximum_payment_date,2.025111e+07
minimum_original_amount,3.633000e+01
maximum_original_amount,4.468870e+05


La fuente contiene 21.739 cuentas por cobrar asociadas a 800 cuentas, 2 productos y 71 códigos de transacción. El valor original suma aproximadamente $141,56 millones y el saldo pendiente $22,19 millones.


## 2. Calidad y granularidad

Se revisan valores nulos, duplicados, montos y coherencia de fechas. También se valida si `num_cta` identifica una obligación individual.


In [47]:
with sqlite3.connect(database_uri, uri=True) as connection:
    quality_checks = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS total_rows,

            SUM(
                CASE
                    WHEN vlr_original < 0
                      OR vlr_pagado < 0
                      OR vlr_pendiente_pago < 0
                    THEN 1
                    ELSE 0
                END
            ) AS rows_with_negative_amounts,

            SUM(
                CASE
                    WHEN ABS(
                        vlr_original
                        - vlr_pagado
                        - vlr_pendiente_pago
                    ) > 0.01
                    THEN 1
                    ELSE 0
                END
            ) AS rows_with_amount_inconsistency,

            SUM(
                CASE
                    WHEN vlr_pagado > vlr_original
                    THEN 1
                    ELSE 0
                END
            ) AS rows_paid_above_original,

            SUM(
                CASE
                    WHEN vlr_pendiente_pago > vlr_original
                    THEN 1
                    ELSE 0
                END
            ) AS rows_pending_above_original
        FROM tabla1;
        """,
        connection,
    )

quality_checks.T

,0
total_rows,21739
rows_with_negative_amounts,0
rows_with_amount_inconsistency,0
rows_paid_above_original,0
rows_pending_above_original,0


In [48]:
column_names = table_schema["name"].tolist()
column_quality = []

with sqlite3.connect(database_uri, uri=True) as connection:
    for column_name in column_names:
        safe_column_name = column_name.replace('"', '""')

        metrics = connection.execute(
            f"""
            SELECT
                COUNT(*) AS total_rows,
                SUM(
                    CASE
                        WHEN "{safe_column_name}" IS NULL THEN 1
                        ELSE 0
                    END
                ) AS null_rows,
                COUNT(DISTINCT "{safe_column_name}") AS distinct_values
            FROM tabla1;
            """
        ).fetchone()

        column_quality.append(
            {
                "column_name": column_name,
                "null_rows": metrics[1],
                "null_percentage": round(
                    metrics[1] / metrics[0] * 100,
                    2,
                ),
                "distinct_values": metrics[2],
            }
        )

column_quality_df = pd.DataFrame(column_quality)
column_quality_df

,column_name,null_rows,null_percentage,distinct_values
0,cod_apli_prod,0,0.0,2
1,descri_cod_apli_prod,0,0.0,2
2,num_cta,0,0.0,800
3,f_creacion,0,0.0,307
4,f_ultimo_pago,0,0.0,378
5,vlr_original,0,0.0,21407
6,vlr_pagado,0,0.0,19696
7,vlr_pendiente_pago,0,0.0,4392
8,cod_trn,0,0.0,71
9,descri_cod_trn,0,0.0,71


In [49]:
with sqlite3.connect(database_uri, uri=True) as connection:
    grain_validation = pd.read_sql_query(
        """
        WITH account_date_grain AS (
            SELECT
                num_cta,
                printf(
                    '%04d-%02d-%02d',
                    year,
                    month,
                    day
                ) AS reference_date,
                COUNT(*) AS row_count
            FROM tabla1
            GROUP BY
                num_cta,
                year,
                month,
                day
        )
        SELECT
            COUNT(*) AS account_date_combinations,
            SUM(
                CASE
                    WHEN row_count > 1 THEN 1
                    ELSE 0
                END
            ) AS duplicated_account_dates,
            MIN(row_count) AS minimum_rows,
            MAX(row_count) AS maximum_rows
        FROM account_date_grain;
        """,
        connection,
    )

grain_validation

,account_date_combinations,duplicated_account_dates,minimum_rows,maximum_rows
0,21739,0,1,1


In [50]:
with sqlite3.connect(database_uri, uri=True) as connection:
    date_validation = pd.read_sql_query(
        """
        WITH prepared_dates AS (
            SELECT
                date(
                    substr(CAST(f_creacion AS TEXT), 1, 4) || '-' ||
                    substr(CAST(f_creacion AS TEXT), 5, 2) || '-' ||
                    substr(CAST(f_creacion AS TEXT), 7, 2)
                ) AS creation_date,

                date(
                    substr(CAST(f_ultimo_pago AS TEXT), 1, 4) || '-' ||
                    substr(CAST(f_ultimo_pago AS TEXT), 5, 2) || '-' ||
                    substr(CAST(f_ultimo_pago AS TEXT), 7, 2)
                ) AS last_payment_date,

                date(
                    printf(
                        '%04d-%02d-%02d',
                        year,
                        month,
                        day
                    )
                ) AS reference_date,

                vlr_pagado
            FROM tabla1
        )
        SELECT
            COUNT(DISTINCT reference_date) AS distinct_reference_dates,
            MIN(reference_date) AS minimum_reference_date,
            MAX(reference_date) AS maximum_reference_date,

            SUM(
                CASE
                    WHEN creation_date > reference_date THEN 1
                    ELSE 0
                END
            ) AS creation_after_reference,

            SUM(
                CASE
                    WHEN last_payment_date > reference_date THEN 1
                    ELSE 0
                END
            ) AS payment_after_reference,

            SUM(
                CASE
                    WHEN last_payment_date < creation_date THEN 1
                    ELSE 0
                END
            ) AS payment_before_creation,

            SUM(
                CASE
                    WHEN vlr_pagado = 0
                     AND last_payment_date IS NOT NULL
                    THEN 1
                    ELSE 0
                END
            ) AS zero_paid_with_payment_date,

            MIN(
                CAST(
                    julianday(reference_date)
                    - julianday(creation_date)
                    AS INTEGER
                )
            ) AS minimum_age_days,

            MAX(
                CAST(
                    julianday(reference_date)
                    - julianday(creation_date)
                    AS INTEGER
                )
            ) AS maximum_age_days
        FROM prepared_dates;
        """,
        connection,
    )

date_validation.T

,0
distinct_reference_dates,32
minimum_reference_date,2025-10-11
maximum_reference_date,2025-11-11
creation_after_reference,0
payment_after_reference,0
payment_before_creation,0
zero_paid_with_payment_date,1747
minimum_age_days,90
maximum_age_days,365


No se identificaron problemas críticos de calidad. `num_cta` se repite y no identifica una cuenta por cobrar única. Por esta razón, se utiliza el `rowid` de SQLite para crear `cxc_id`.

Las columnas `year`, `month` y `day` se interpretan como fecha de referencia porque la fuente no incluye un diccionario de datos. Este supuesto queda documentado.


## 3. Estado de pago y recuperación

Se construye una clasificación observada con tres estados: pago total, pago parcial y sin pago. La tolerancia de 0,01 controla diferencias de redondeo monetario.


In [51]:
prepared_data_query = """
WITH normalized_dates AS (
    SELECT
        rowid AS source_row_id,
        cod_apli_prod,
        descri_cod_apli_prod,
        num_cta,
        cod_trn,
        descri_cod_trn,
        vlr_original,
        vlr_pagado,
        vlr_pendiente_pago,

        date(
            substr(CAST(f_creacion AS TEXT), 1, 4) || '-' ||
            substr(CAST(f_creacion AS TEXT), 5, 2) || '-' ||
            substr(CAST(f_creacion AS TEXT), 7, 2)
        ) AS creation_date,

        date(
            substr(CAST(f_ultimo_pago AS TEXT), 1, 4) || '-' ||
            substr(CAST(f_ultimo_pago AS TEXT), 5, 2) || '-' ||
            substr(CAST(f_ultimo_pago AS TEXT), 7, 2)
        ) AS last_payment_date,

        date(
            printf(
                '%04d-%02d-%02d',
                year,
                month,
                day
            )
        ) AS reference_date
    FROM tabla1
)
SELECT
    *,
    CAST(
        julianday(reference_date)
        - julianday(creation_date)
        AS INTEGER
    ) AS age_days,

    CAST(
        julianday(reference_date)
        - julianday(last_payment_date)
        AS INTEGER
    ) AS days_since_last_payment,

    ROUND(
        vlr_pagado / NULLIF(vlr_original, 0),
        4
    ) AS recovery_rate,

    ROUND(
        vlr_pendiente_pago / NULLIF(vlr_original, 0),
        4
    ) AS pending_rate,

    CASE
        WHEN vlr_pendiente_pago <= 0.01
            THEN 'pagada_total'
        WHEN vlr_pagado <= 0.01
            THEN 'sin_pago'
        ELSE 'pago_parcial'
    END AS payment_status
FROM normalized_dates;
"""

with sqlite3.connect(database_uri, uri=True) as connection:
    prepared_data = pd.read_sql_query(
        prepared_data_query,
        connection,
    )

prepared_data.shape

(21739, 17)

In [52]:
status_summary = (
    prepared_data
    .groupby("payment_status", as_index=False)
    .agg(
        records=("source_row_id", "count"),
        original_amount=("vlr_original", "sum"),
        paid_amount=("vlr_pagado", "sum"),
        pending_amount=("vlr_pendiente_pago", "sum"),
    )
)

status_summary["record_percentage"] = (
    status_summary["records"]
    / status_summary["records"].sum()
    * 100
).round(2)

status_summary["weighted_recovery_rate"] = (
    status_summary["paid_amount"]
    / status_summary["original_amount"]
    * 100
).round(2)

status_summary

,payment_status,records,original_amount,paid_amount,pending_amount,record_percentage,weighted_recovery_rate
0,pagada_total,17323,1.059582e+08,1.059582e+08,0.00,79.69,100.00
1,pago_parcial,2669,2.149606e+07,1.341643e+07,8079629.05,12.28,62.41
2,sin_pago,1747,1.411037e+07,0.000000e+00,14110371.25,8.04,0.00


In [53]:
product_summary = (
    prepared_data
    .groupby(
        [
            "cod_apli_prod",
            "descri_cod_apli_prod",
        ],
        as_index=False,
    )
    .agg(
        records=("source_row_id", "count"),
        distinct_accounts=("num_cta", "nunique"),
        original_amount=("vlr_original", "sum"),
        paid_amount=("vlr_pagado", "sum"),
        pending_amount=("vlr_pendiente_pago", "sum"),
    )
)

product_summary["weighted_recovery_rate"] = (
    product_summary["paid_amount"]
    / product_summary["original_amount"]
    * 100
).round(2)

product_summary

,cod_apli_prod,descri_cod_apli_prod,records,distinct_accounts,original_amount,paid_amount,pending_amount,weighted_recovery_rate
0,D,CORRIENTE,248,9,6.977096e+05,6.831060e+05,14603.58,97.91
1,S,AHORRO,21491,791,1.408669e+08,1.186915e+08,22175396.72,84.26


El 79,69 % de las obligaciones está pagado totalmente. Los pagos parciales representan 12,28 % de los registros y mantienen una recuperación ponderada de 62,41 %, por lo que constituyen un grupo relevante para seguimiento.

AHORRO concentra casi toda la población. CORRIENTE muestra mayor recuperación, pero solo incluye nueve cuentas; la diferencia es descriptiva y no permite concluir que el producto tenga mejor comportamiento.


In [54]:
transaction_summary = (
    prepared_data
    .groupby(
        [
            "cod_trn",
            "descri_cod_trn",
        ],
        as_index=False,
    )
    .agg(
        records=("source_row_id", "count"),
        distinct_accounts=("num_cta", "nunique"),
        original_amount=("vlr_original", "sum"),
        paid_amount=("vlr_pagado", "sum"),
        pending_amount=("vlr_pendiente_pago", "sum"),
    )
)

transaction_summary["weighted_recovery_rate"] = (
    transaction_summary["paid_amount"]
    / transaction_summary["original_amount"]
    * 100
).round(2)

transaction_summary["pending_share"] = (
    transaction_summary["pending_amount"]
    / transaction_summary["pending_amount"].sum()
    * 100
).round(2)

transaction_summary = transaction_summary.sort_values(
    "pending_amount",
    ascending=False,
)

transaction_summary.head(10)

,cod_trn,descri_cod_trn,records,distinct_accounts,original_amount,paid_amount,pending_amount,weighted_recovery_rate,pending_share
33,834,CARGO FISCAL TRANSACCIONAL,6823,250,36024417.71,30466166.84,5558250.87,84.57,25.05
44,1517,COBRO SERVICIO TRANSPORTE,3704,137,18798035.98,15411102.47,3386933.51,81.98,15.26
60,2106,COMISION TRANSFERENCIA EXTERNA B,1184,44,9401013.71,6625590.42,2775423.29,70.48,12.51
8,176,COMISION RETIRO CORRESPONSAL B,1129,41,11470689.41,8875909.88,2594779.53,77.38,11.69
6,137,COMISION RETIRO CANAL A,645,24,6454303.32,4597445.02,1856858.30,71.23,8.37
36,1157,COMISION CONSULTA SALDO,802,30,5606796.26,3932023.32,1674772.94,70.13,7.55
24,497,PAGO SERVICIO ELECTRONICO,1312,48,8782114.18,7399279.46,1382834.72,84.25,6.23
4,87,CARGO FISCAL IVA COMISION B,659,24,2913324.76,2372884.01,540440.75,81.45,2.44
30,788,TRANSFERENCIA CANAL FISICO,139,5,688603.78,207814.61,480789.17,30.18,2.17
61,2107,CARGO FISCAL IVA TRASLADO,1070,40,5327006.77,4915008.96,411997.81,92.27,1.86


El saldo pendiente se concentra en pocos códigos de transacción. Estas diferencias justifican incluir `cod_trn` en el análisis, pero los códigos con pocos registros deben interpretarse con cautela.


## 4. Antigüedad y montos

La antigüedad se calcula desde la fecha de creación hasta la fecha de referencia. No representa días de mora porque la fuente no contiene fecha de vencimiento.


In [55]:
date_columns = [
    "creation_date",
    "last_payment_date",
    "reference_date",
]

for column_name in date_columns:
    prepared_data[column_name] = pd.to_datetime(
        prepared_data[column_name],
        errors="coerce",
    )

In [56]:
age_bins = [
    float("-inf"),
    30,
    60,
    90,
    180,
    360,
    float("inf"),
]

age_labels = [
    "00_30_dias",
    "31_60_dias",
    "61_90_dias",
    "91_180_dias",
    "181_360_dias",
    "361_mas_dias",
]

prepared_data["age_bucket"] = pd.cut(
    prepared_data["age_days"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True,
)

prepared_data["is_fully_paid"] = (
    prepared_data["payment_status"] == "pagada_total"
).astype(int)

prepared_data["is_unpaid"] = (
    prepared_data["payment_status"] == "sin_pago"
).astype(int)

In [57]:
age_summary = (
    prepared_data
    .groupby(
        "age_bucket",
        observed=False,
        as_index=False,
    )
    .agg(
        records=("source_row_id", "count"),
        distinct_accounts=("num_cta", "nunique"),
        original_amount=("vlr_original", "sum"),
        paid_amount=("vlr_pagado", "sum"),
        pending_amount=("vlr_pendiente_pago", "sum"),
        fully_paid_records=("is_fully_paid", "sum"),
        unpaid_records=("is_unpaid", "sum"),
    )
)

age_summary["weighted_recovery_rate"] = (
    age_summary["paid_amount"]
    / age_summary["original_amount"]
    * 100
).round(2)

age_summary["fully_paid_rate"] = (
    age_summary["fully_paid_records"]
    / age_summary["records"]
    * 100
).round(2)

age_summary["unpaid_rate"] = (
    age_summary["unpaid_records"]
    / age_summary["records"]
    * 100
).round(2)

age_summary

,age_bucket,records,distinct_accounts,original_amount,paid_amount,pending_amount,fully_paid_records,unpaid_records,weighted_recovery_rate,fully_paid_rate,unpaid_rate
0,00_30_dias,0,0,0.00,0.00,0.00,0,0,NaN,NaN,NaN
1,31_60_dias,0,0,0.00,0.00,0.00,0,0,NaN,NaN,NaN
2,61_90_dias,82,76,480466.18,401312.96,79153.22,67,7,83.53,81.71,8.54
3,91_180_dias,7124,800,47966336.55,40368275.52,7598061.03,5666,590,84.16,79.53,8.28
4,181_360_dias,14150,800,90504618.73,76445458.21,14059160.52,11283,1116,84.47,79.74,7.89
5,361_mas_dias,383,304,2613227.78,2159602.25,453625.53,307,34,82.64,80.16,8.88


In [58]:
prepared_data["age_days"].describe()

count    21739.000000
mean       226.844059
std         79.303880
min         90.000000
25%        158.000000
50%        226.000000
75%        295.000000
max        365.000000
Name: age_days, dtype: float64

La recuperación no presenta una relación monotónica fuerte con la antigüedad. Además, no existen registros con menos de 61 días, lo que limita el análisis del comportamiento inicial.


In [59]:
amount_summary = (
    prepared_data[
        [
            "vlr_original",
            "vlr_pagado",
            "vlr_pendiente_pago",
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .T
)

amount_summary

,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max
vlr_original,21739.0,6512.012937,19140.521465,36.33,72.9976,198.989,856.605,2218.19,6001.660,13580.200,23666.847,57126.2476,446887.02
vlr_pagado,21739.0,5491.266799,17918.782539,0.00,0.0000,0.000,567.025,1782.25,5041.595,12388.988,19589.441,44773.9448,446887.02
vlr_pendiente_pago,21739.0,1020.746138,5955.033068,0.00,0.0000,0.000,0.000,0.00,0.000,1280.428,4415.696,20174.2970,173830.09


In [60]:
prepared_data["amount_bucket"] = pd.qcut(
    prepared_data["vlr_original"],
    q=5,
    labels=[
        "muy_bajo",
        "bajo",
        "medio",
        "alto",
        "muy_alto",
    ],
    duplicates="drop",
)

In [61]:
amount_bucket_summary = (
    prepared_data
    .groupby(
        "amount_bucket",
        observed=False,
        as_index=False,
    )
    .agg(
        records=("source_row_id", "count"),
        distinct_accounts=("num_cta", "nunique"),
        minimum_amount=("vlr_original", "min"),
        maximum_amount=("vlr_original", "max"),
        original_amount=("vlr_original", "sum"),
        paid_amount=("vlr_pagado", "sum"),
        pending_amount=("vlr_pendiente_pago", "sum"),
        fully_paid_records=("is_fully_paid", "sum"),
        unpaid_records=("is_unpaid", "sum"),
    )
)

amount_bucket_summary["weighted_recovery_rate"] = (
    amount_bucket_summary["paid_amount"]
    / amount_bucket_summary["original_amount"]
    * 100
).round(2)

amount_bucket_summary["fully_paid_rate"] = (
    amount_bucket_summary["fully_paid_records"]
    / amount_bucket_summary["records"]
    * 100
).round(2)

amount_bucket_summary["unpaid_rate"] = (
    amount_bucket_summary["unpaid_records"]
    / amount_bucket_summary["records"]
    * 100
).round(2)

amount_bucket_summary

,amount_bucket,records,distinct_accounts,minimum_amount,maximum_amount,original_amount,paid_amount,pending_amount,fully_paid_records,unpaid_records,weighted_recovery_rate,fully_paid_rate,unpaid_rate
0,muy_bajo,4348,187,36.33,684.18,1.565601e+06,1400912.25,164688.60,3546,298,89.48,81.55,6.85
1,bajo,4348,216,684.22,1571.59,4.769041e+06,4185757.65,583283.46,3397,325,87.77,78.13,7.47
2,medio,4347,218,1571.73,3259.68,9.947335e+06,8924667.17,1022668.14,3636,297,89.72,83.64,6.83
3,alto,4348,218,3261.07,7407.33,2.198205e+07,18827668.59,3154376.55,3381,403,85.65,77.76,9.27
4,muy_alto,4348,187,7407.97,446887.02,1.033006e+08,86035643.28,17264983.55,3363,424,83.29,77.35,9.75


El quintil de mayor valor presenta la menor recuperación ponderada y concentra una parte importante del saldo. El valor original puede aportar información para la priorización y la modelación, aunque los quintiles no representan umbrales operativos definitivos.


## 5. Granularidad final y diseño de la sábana

Cada registro representa una cuenta por cobrar individual. `num_cta` identifica la cuenta asociada, pero puede repetirse en diferentes obligaciones.


In [62]:
account_consistency = (
    prepared_data
    .groupby("num_cta", as_index=False)
    .agg(
        observations=("reference_date", "nunique"),
        first_reference_date=("reference_date", "min"),
        last_reference_date=("reference_date", "max"),
        distinct_products=("cod_apli_prod", "nunique"),
        distinct_transactions=("cod_trn", "nunique"),
        distinct_creation_dates=("creation_date", "nunique"),
        distinct_original_amounts=("vlr_original", "nunique"),
    )
)

consistency_summary = pd.DataFrame(
    {
        "validation": [
            "accounts",
            "multiple_products",
            "multiple_transactions",
            "multiple_creation_dates",
            "multiple_original_amounts",
            "minimum_observations",
            "maximum_observations",
        ],
        "value": [
            len(account_consistency),
            (
                account_consistency["distinct_products"] > 1
            ).sum(),
            (
                account_consistency["distinct_transactions"] > 1
            ).sum(),
            (
                account_consistency["distinct_creation_dates"] > 1
            ).sum(),
            (
                account_consistency["distinct_original_amounts"] > 1
            ).sum(),
            account_consistency["observations"].min(),
            account_consistency["observations"].max(),
        ],
    }
)

consistency_summary

,validation,value
0,accounts,800
1,multiple_products,0
2,multiple_transactions,0
3,multiple_creation_dates,800
4,multiple_original_amounts,800
5,minimum_observations,19
6,maximum_observations,32


La transformación sigue cuatro pasos: fuente original, normalización en SQL, sábana analítica y vista de modelación.

La sábana conserva variables originales y derivadas para análisis y monitoreo. Para el modelo se excluyen variables posteriores al pago, como `vlr_pagado`, `vlr_pendiente_pago`, `last_payment_date`, `payment_status` y `recovery_rate`, porque producirían fuga de información. `num_cta` se utiliza para separar entrenamiento y validación, no como predictor directo.


In [63]:
from src.analytical_table_builder import AnalyticalTableBuilder
from src.database import SQLiteDatabase


database = SQLiteDatabase(database_path)

table_builder = AnalyticalTableBuilder(
    database=database,
    sql_path=project_path / "src" / "sql" / "sabana_analitica.sql",
)

analytical_table = table_builder.build()
table_builder.validate(analytical_table)

,validation,value
0,total_rows,21739
1,unique_cxc_ids,21739
2,duplicated_cxc_ids,0
3,inconsistent_amount_rows,0


In [64]:
analytical_output_path = (
    project_path
    / "data"
    / "processed"
    / "sabana_analitica.csv"
)

table_builder.export_csv(
    analytical_table=analytical_table,
    output_path=analytical_output_path,
)

print("Sábana exportada: data/processed/sabana_analitica.csv")

Sábana exportada: data/processed/sabana_analitica.csv


## Conclusión

La fuente tiene calidad suficiente para continuar. La unidad de análisis es una cuenta por cobrar individual, identificada mediante `cxc_id`. La sábana integra variables originales y derivadas y deja documentadas las restricciones necesarias para construir el modelo sin fuga de información.
